This notebook processes the two replicates of the cohen retina scMPRA dataset into an ortho object.

# Setup

In [1]:
import scMPRAforge as scm

In [2]:
#create dask cluster

#from dask_jobqueue import SLURMCluster
#from dask.distributed import Client

#cluster=SLURMCluster(
#    cores=4,#cores per slurm job
#    memory="64G",#memory per slurm job
#    processes=1,#dask workers per slurm job
#    job_extra_directives=["-p ycga", 
#        f"--job-name=simclust_worker",
#        f"--time=9:00:00",
#        f"--output=slave_%j.out"]
#)
#
#cluster.scale(jobs=6)
#
#client = Client(cluster,
#        timeout=f"{5*60}s",   # Client <-> scheduler timeout 
#        heartbeat_interval="30s"  # Worker heartbeat interval
#    )

from dask.distributed import Client, LocalCluster
cluster=LocalCluster(memory_limit='24GB')
client = Client(cluster)

# Describe with an ortho

Regarding reference cell-type,
> Reproducibility was highest in rod cells (Spearman’s ρ = 0.97, Pearson’s R = 0.98) because rod cells are the most abundant cell type in the mouse retina.

Regarding negative controls 
> In this experiment, we define the effect of a mutation as its relative fold-change to the WT Gnb3 promoter in each cell type because the Gnb3 promoter is expressed at different levels across cell types (Fig. 5a).

Interestingly, there are two wt promoters. Some code in the cohen paper's zenodo (`Part2_section6_retina_analysis.ipynb`) suggests that they should be treated together:

```
rep1_wt_df = rep1_exp.loc[rep1_exp['name'].str.contains('wt')]
mean_exp = np.mean(rep1_wt_df['mean'])
rep1_exp['activity'] = rep1_exp['mean']/mean_exp
```

So we will simply lump both, despite the fact that they have different sequences.

In [3]:
data_root="/gpfs/gibbs/pi/reilly/tabula_data"
path="/gpfs/gibbs/pi/reilly/tabula_data/cohen"
name="ortho_primordial"


import os
if os.path.isdir(path+"/"+name):
    print("[+] Model found. Loading...")
    primordial=scm.ortho.load(client,path,name)
else:
    print("[+] Model not found. Creating...")

    #load data
    cohen=scm.scMPRA_data.from_parquet(f"{path}/retina_single_counting_u6.scmpra")
    cohen.set_negative_controls(["wt_1","wt_2"])
    cohen.set_reference_cell("Rod")
    cohen.ortho_filter()

    primordial=scm.ortho()
    primordial.criss_cross(client=client,
                       dat=cohen)
    primordial.extract_params(client)
    #primordial.save(path,name)


[+] Model not found. Creating...


scMPRAforge: INFO: Dropped 0 of 456 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


Metal device set to: Apple M4 Pro

systemMemory: 24.00 GB
maxCacheSize: 8.00 GB

Metal device set to: Apple M4 Pro

systemMemory: 24.00 GB
maxCacheSize: 8.00 GB

Metal device set to: Apple M4 Pro

systemMemory: 24.00 GB
maxCacheSize: 8.00 GB

Metal device set to: Apple M4 Pro

systemMemory: 24.00 GB
maxCacheSize: 8.00 GB

Metal device set to: Apple M4 Pro

systemMemory: 24.00 GB
maxCacheSize: 8.00 GB

Metal device set to: Apple M4 Pro

systemMemory: 24.00 GB
maxCacheSize: 8.00 GB

Metal device set to: Apple M4 Pro

systemMemory: 24.00 GB
maxCacheSize: 8.00 GB



In [6]:
primordial.by_cell_type.model

{'reference': <Future: pending, key: _label_tensorzinb_regressors-f923aaf4645d9fc3a5ac7cd5c97115e2>,
 'Mueller Glia': <Future: pending, key: _label_tensorzinb_regressors-795cf4b7408580de31422c5559e8780b>,
 'Interneuron': <Future: pending, key: _label_tensorzinb_regressors-6730a520a08173dd342fcf49c3bfa1bd>,
 'Bipolar': <Future: pending, key: _label_tensorzinb_regressors-4c9b185ef2d58cb3a665362cd7fc8f41>}

In [5]:
client.dashboard_link

'http://127.0.0.1:55057/status'

In [ ]:
client.who_has(primordial.by_cell_type.model['Interneuron'])

In [ ]:
client.who_has(primordial.by_cell_type.model['reference'])

In [ ]:
primordial.by_cre.model

In [ ]:
cluster.close()
client.close()

Examining QC metrics : mean VS estimated parameter